# Notebook 03_1 — Predictive Performance (Text + Image Embeddings)

**Replication of Bach et al. (2025) — Adventures in Demand Analysis Using AI**  
Applied to: Amazon Women's Shoes (Size 8)

---

Same as 03_2 but uses **multimodal embeddings** (RoBERTa + BEiT + SAINT).  
Adds `x_emb` as a fourth feature specification on top of x, x_pca, x_sim.

Model specifications compared:
- OLS / Boosting — Tabular only
- OLS / Boosting — Tabular + PCA
- OLS / Boosting — Tabular + Similarities
- OLS / Boosting — Tabular + Embeddings (multimodal)
- Deep Time Independent
- Deep Time Dependent

## ① Mount Drive

In [1]:
# Local mode - no Google Drive needed
print('Local mode')

Local mode


## ② Set Working Directory

In [2]:
import os, sys
from pathlib import Path

# Local mode - no Google Drive needed
_cwd = Path.cwd()
PROJECT_ROOT = _cwd
for _ in range(5):
    if (PROJECT_ROOT / 'data').is_dir() and (PROJECT_ROOT / 'code').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Cannot find project root")

CODE_DIR = str(PROJECT_ROOT / 'code')
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print(f'Working directory: {os.getcwd()}')

Working directory: /home/iankuzuma/claude_code/demand_modeling/women-8-subcat-split-proper-embedding/pumps/code


## ③ Imports

In [3]:
import re
import datasets
import pandas as pd
import numpy as np
import statsmodels.api as sm

from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score

from utils.utils_data2 import (
    load_pred_and_emb,
    center_and_norm,
    get_pca,
    get_cluster,
    get_similarities,
    add_lags_and_scale_data,
)
print('✅ Imports done')

✅ Imports done


## ④ Load Dataset

Loads the **multimodal** embeddings dataset from notebook 01_1 (txt_only=False).
Key difference from 03_2: this dataset includes both text AND image embeddings.

In [4]:
txt_only = False
embeddings = True
dataframe_name = f"dataset_txt_only_{txt_only}_embeddings_{embeddings}"

df_full_train = pd.read_csv(f"../data/{dataframe_name}_train.zip")
df_full_val   = pd.read_csv(f"../data/{dataframe_name}_val.zip")

columns_to_drop = [
    "Q_t-2", "P_bb_t-2", "REVIEW_COUNT_t-2", "RATING_t-2",
]
df_full_train = df_full_train.drop(columns=columns_to_drop)
df_full_val   = df_full_val.drop(columns=columns_to_drop)

df_full_train = df_full_train.dropna()
df_full_val   = df_full_val.dropna()

dummy_subcat_names = [category for category in df_full_val["subcat_aggregated"].unique()]
all_time_steps = sorted([str(date) for date in df_full_val["date"].unique()])

print(f"Train shape: {df_full_train.shape}")
print(f"Val shape:   {df_full_val.shape}")
print(f"Dummy subcat names: {dummy_subcat_names}")
print(f"All time steps: {all_time_steps}")
df_full_val.columns

Train shape: (7512, 323)
Val shape:   (7524, 323)
Dummy subcat names: ['Pumps']
All time steps: ['2025-04-28', '2025-05-26', '2025-06-23', '2025-07-21', '2025-08-18', '2025-09-15', '2025-10-13', '2025-11-10', '2025-12-08', '2026-01-05', '2026-02-02', '2026-03-02']


Index(['ASIN', 'date', 'Q_t', 'PRICE', 'P_bb_t', 'text', 'window',
       'REVIEW_COUNT', 'RATING', 'New Offer Count: Current',
       ...
       'Delta_Q_t', 'Delta_P_bb_t', 'pred_ml_l', 'pred_ml_m', 'pred_ml_l_diff',
       'pred_ml_m_diff', 'pred_ml_l_lag_1', 'pred_ml_m_lag_1',
       'pred_ml_l_diff_lag_1', 'pred_ml_m_diff_lag_1'],
      dtype='object', length=323)

## ⑤ Sanity Check — Row Counts

In [5]:
print(f"Val rows:   {len(df_full_val)}")
print(f"Train rows: {len(df_full_train)}")

Val rows:   7524
Train rows: 7512


## ⑥ Define Controls and Feature Specifications

Same controls as 03_2 plus `controls_emb` — all 256 multimodal embedding dimensions.

In [6]:
n_lags = 1

outcome   = "Q_t"
treatment = "P_bb_t"

outcome_diff   = "Delta_Q_t"
treatment_diff = "Delta_P_bb_t"

dummy_time_steps = all_time_steps[n_lags:]

all_dummy_controls = (
    dummy_subcat_names
    + dummy_time_steps
    + ["Lightning Deals: Upcoming Deal", "Buy Box: Is FBA"]
)

dummy_baselines = [dummy_time_steps[0], "Residual"]
dummy_time_steps_wo_baseline = [t for t in dummy_time_steps if t not in dummy_baselines]
dummy_controls = [t for t in all_dummy_controls if t not in dummy_baselines]

cont_controls = [
    "RATING_t-1",
    "REVIEW_COUNT_t-1",
    "New Offer Count: Current",
    "Count of retrieved live offers: New, FBA",
    "Count of retrieved live offers: New, FBM",
]

add_controls_to_scale = ["RATING", "REVIEW_COUNT"]

controls_pca = ["pca_0", "pca_1", "pca_2", "pca_3", "pca_4"]
controls_similarities = [
    "similarity_cluster_0", "similarity_cluster_1", "similarity_cluster_2",
    "similarity_cluster_3", "similarity_cluster_4",
]

additional_controls = cont_controls + dummy_controls
additional_controls_deep = [var for var in additional_controls if var not in dummy_subcat_names]

# Key difference from 03_2 — multimodal emb columns
controls_emb = [var for var in df_full_train.columns if "emb" in var]

print(f"Continuous controls:    {len(cont_controls)}")
print(f"Dummy controls:         {len(dummy_controls)}")
print(f"Total controls:         {len(additional_controls)}")
print(f"Embedding columns:      {len(controls_emb)}")

Continuous controls:    5
Dummy controls:         13
Total controls:         18
Embedding columns:      256


## ⑦ Initialize Results DataFrames

In [7]:
column_names = ["R2 Q Train", "R2 Q Test", "R2 P Train", "R2 P Test"]

results_df      = pd.DataFrame(columns=column_names)
results_df_diff = pd.DataFrame(columns=column_names)
print('✅ Results DataFrames initialized')

✅ Results DataFrames initialized


## ⑧ Deep Model R² — Level

In [8]:
df_dict = {"Train": df_full_train, "Test": df_full_val}

results_df_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y = df[outcome].values
    d = df[treatment].values

    pred_ml_l = df["pred_ml_l"].values
    pred_ml_m = df["pred_ml_m"].values

    r2_ml_l = np.round(r2_score(y, pred_ml_l), 4)
    r2_ml_m = np.round(r2_score(d, pred_ml_m), 4)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome (time independent):   {r2_ml_l}")
    print(f"  R2 Treatment (time independent):  {r2_ml_m}")

    pred_ml_l_lag1 = df["pred_ml_l_lag_1"].values
    pred_ml_m_lag1 = df["pred_ml_m_lag_1"].values

    r2_ml_l_lag1 = np.round(r2_score(y, pred_ml_l_lag1), 4)
    r2_ml_m_lag1 = np.round(r2_score(d, pred_ml_m_lag1), 4)

    print(f"  R2 Outcome (lag1):                {r2_ml_l_lag1}")
    print(f"  R2 Treatment (lag1):              {r2_ml_m_lag1}")
    print()

    results_df_deep[f"R2 Q {df_name}"] = (r2_ml_l, r2_ml_l_lag1)
    results_df_deep[f"R2 P {df_name}"] = (r2_ml_m, r2_ml_m_lag1)

results_df_deep

Evaluation for Train set
  R2 Outcome (time independent):   0.7981
  R2 Treatment (time independent):  -0.0012
  R2 Outcome (lag1):                0.8632
  R2 Treatment (lag1):              0.001

Evaluation for Test set
  R2 Outcome (time independent):   0.6695
  R2 Treatment (time independent):  -0.0674
  R2 Outcome (lag1):                0.8085
  R2 Treatment (lag1):              -0.0846



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.7981,0.6695,-0.0012,-0.0674
Deep Time Dependent,0.8632,0.8085,0.0010,-0.0846


## ⑨ Deep Model R² — Diff

In [9]:
results_df_diff_deep = pd.DataFrame(
    data=np.full((2, 4), np.nan),
    columns=column_names,
    index=["Deep Time Independent", "Deep Time Dependent"],
)

for df_name, df in df_dict.items():
    y_diff = df["Delta_Q_t"].values
    d_diff = df["Delta_P_bb_t"].values

    pred_ml_l_diff = df["pred_ml_l_diff"].values
    pred_ml_m_diff = df["pred_ml_m_diff"].values

    r2_ml_l_diff = np.round(r2_score(y_diff, pred_ml_l_diff), 8)
    r2_ml_m_diff = np.round(r2_score(d_diff, pred_ml_m_diff), 8)

    print(f"Evaluation for {df_name} set")
    print(f"  R2 Outcome diff (time independent):   {r2_ml_l_diff}")
    print(f"  R2 Treatment diff (time independent):  {r2_ml_m_diff}")

    pred_ml_l_diff_lag1 = df["pred_ml_l_diff_lag_1"].values
    pred_ml_m_diff_lag1 = df["pred_ml_m_diff_lag_1"].values

    r2_ml_l_diff_lag1 = np.round(r2_score(y_diff, pred_ml_l_diff_lag1), 8)
    r2_ml_m_diff_lag1 = np.round(r2_score(d_diff, pred_ml_m_diff_lag1), 8)

    print(f"  R2 Outcome diff (lag1):                {r2_ml_l_diff_lag1}")
    print(f"  R2 Treatment diff (lag1):              {r2_ml_m_diff_lag1}")
    print()

    results_df_diff_deep[f"R2 Q {df_name}"] = (r2_ml_l_diff, r2_ml_l_diff_lag1)
    results_df_diff_deep[f"R2 P {df_name}"] = (r2_ml_m_diff, r2_ml_m_diff_lag1)

results_df_diff_deep

Evaluation for Train set
  R2 Outcome diff (time independent):   0.00472604
  R2 Treatment diff (time independent):  -0.01974994
  R2 Outcome diff (lag1):                0.01239317
  R2 Treatment diff (lag1):              -0.01563572

Evaluation for Test set
  R2 Outcome diff (time independent):   0.00141142
  R2 Treatment diff (time independent):  -0.01722007
  R2 Outcome diff (lag1):                0.01006396
  R2 Treatment diff (lag1):              -0.01357359



,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
Deep Time Independent,0.004726,0.001411,-0.019750,-0.017220
Deep Time Dependent,0.012393,0.010064,-0.015636,-0.013574


## ⑩ Build Feature Matrices

Key difference from 03_2: adds `x_train_emb` and `x_test_emb`
using all 256 multimodal embedding dimensions.

In [10]:
# Train set
y_train      = df_full_train[outcome].squeeze()
d_train      = df_full_train[treatment].squeeze()
y_train_diff = df_full_train[outcome_diff].squeeze()
d_train_diff = df_full_train[treatment_diff].squeeze()

x_train     = sm.add_constant(df_full_train[additional_controls])
x_train_pca = sm.add_constant(df_full_train[additional_controls + controls_pca])
x_train_sim = sm.add_constant(df_full_train[additional_controls + controls_similarities])
x_train_emb = sm.add_constant(df_full_train[additional_controls + controls_emb])

# Test set
y_test      = df_full_val[outcome].squeeze()
d_test      = df_full_val[treatment].squeeze()
y_test_diff = df_full_val[outcome_diff].squeeze()
d_test_diff = df_full_val[treatment_diff].squeeze()

x_test     = sm.add_constant(df_full_val[additional_controls])
x_test_pca = sm.add_constant(df_full_val[additional_controls + controls_pca])
x_test_sim = sm.add_constant(df_full_val[additional_controls + controls_similarities])
x_test_emb = sm.add_constant(df_full_val[additional_controls + controls_emb])

# Rename columns for LightGBM
for df in [x_train, x_test, x_train_pca, x_test_pca,
           x_train_sim, x_test_sim, x_train_emb, x_test_emb]:
    df.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x), inplace=True)

print(f"x_train shape:     {x_train.shape}")
print(f"x_train_pca shape: {x_train_pca.shape}")
print(f"x_train_sim shape: {x_train_sim.shape}")
print(f"x_train_emb shape: {x_train_emb.shape}")

x_train shape:     (7512, 18)
x_train_pca shape: (7512, 23)
x_train_sim shape: (7512, 23)
x_train_emb shape: (7512, 274)


## ⑪ Build Dict Structures

In [11]:
train_dict = {
    "y": y_train, "y_diff": y_train_diff,
    "d": d_train, "d_diff": d_train_diff,
    "x": x_train, "x_pca": x_train_pca,
    "x_sim": x_train_sim, "x_emb": x_train_emb,
}

test_dict = {
    "y": y_test, "y_diff": y_test_diff,
    "d": d_test, "d_diff": d_test_diff,
    "x": x_test, "x_pca": x_test_pca,
    "x_sim": x_test_sim, "x_emb": x_test_emb,
}
print('✅ Train and test dicts ready')

✅ Train and test dicts ready


## ⑫ Tabular Models — Level

Four feature specifications including the new x_emb (Tabular + full embeddings).

In [12]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y"])
    print(f"  R2 Outcome train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d"])
    print(f"  R2 Treatment train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_tab = pd.concat([results_df_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_tab.columns = column_names
results_df_tab.index   = row_names
results_df_tab


 Feature specification: x
  OLS
  R2 Outcome train/test: 0.2624 / 0.2888
  R2 Treatment train/test: 0.0390 / 0.0251
  Boosting


  R2 Outcome train/test: 0.7136 / 0.6308
  R2 Treatment train/test: 0.4498 / 0.0766

 Feature specification: x_pca
  OLS
  R2 Outcome train/test: 0.6826 / 0.5880
  R2 Treatment train/test: 0.1046 / 0.0378
  Boosting


  R2 Outcome train/test: 0.9346 / 0.7401


  R2 Treatment train/test: 0.7369 / 0.0414

 Feature specification: x_sim
  OLS
  R2 Outcome train/test: 0.6816 / 0.5918
  R2 Treatment train/test: 0.1050 / 0.0391
  Boosting


  R2 Outcome train/test: 0.9326 / 0.7393


  R2 Treatment train/test: 0.7393 / 0.1472

 Feature specification: x_emb
  OLS
  R2 Outcome train/test: 0.8355 / 0.5269
  R2 Treatment train/test: 0.3796 / -0.3997
  Boosting


  R2 Outcome train/test: 0.9690 / 0.7519


  R2 Treatment train/test: 0.8896 / 0.0668


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.262367,0.288840,0.039006,0.025097
Boosting (Tabular),0.713553,0.630780,0.449753,0.076600
OLS (Tabular + PCA),0.682643,0.588035,0.104588,0.037784
Boosting (Tabular + PCA),0.934635,0.740118,0.736885,0.041365
OLS (Tabular + Similarities),0.681603,0.591798,0.105034,0.039078
Boosting (Tabular + Similarities),0.932563,0.739290,0.739293,0.147188
OLS (Tabular + Embeddings),0.835520,0.526918,0.379582,-0.399718
Boosting (Tabular + Embeddings),0.968962,0.751895,0.889640,0.066800


## ⑬ Summary — Level Models

In [13]:
results_df = pd.concat([results_df_tab, results_df_deep], axis=0)
results_df

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.262367,0.288840,0.039006,0.025097
Boosting (Tabular),0.713553,0.630780,0.449753,0.076600
OLS (Tabular + PCA),0.682643,0.588035,0.104588,0.037784
Boosting (Tabular + PCA),0.934635,0.740118,0.736885,0.041365
OLS (Tabular + Similarities),0.681603,0.591798,0.105034,0.039078
Boosting (Tabular + Similarities),0.932563,0.739290,0.739293,0.147188
OLS (Tabular + Embeddings),0.835520,0.526918,0.379582,-0.399718
Boosting (Tabular + Embeddings),0.968962,0.751895,0.889640,0.066800
Deep Time Independent,0.798100,0.669500,-0.001200,-0.067400
Deep Time Dependent,0.863200,0.808500,0.001000,-0.084600


## ⑭ Tabular Models — Diff

In [14]:
feature_specifications = ["x", "x_pca", "x_sim", "x_emb"]
results_df_diff_tab = pd.DataFrame()

for feature_specification in feature_specifications:
    print(f"\n Feature specification: {feature_specification}")
    print("  OLS")
    outcome_reg = sm.OLS(train_dict["y_diff"], train_dict[feature_specification]).fit()
    ols_outcome_train   = r2_score(train_dict["y_diff"], outcome_reg.predict(train_dict[feature_specification]))
    ols_outcome_test    = r2_score(test_dict["y_diff"],  outcome_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Outcome diff train/test: {ols_outcome_train:.4f} / {ols_outcome_test:.4f}")

    treatment_reg = sm.OLS(train_dict["d_diff"], train_dict[feature_specification]).fit()
    ols_treatment_train = r2_score(train_dict["d_diff"], treatment_reg.predict(train_dict[feature_specification]))
    ols_treatment_test  = r2_score(test_dict["d_diff"],  treatment_reg.predict(test_dict[feature_specification]))
    print(f"  R2 Treatment diff train/test: {ols_treatment_train:.4f} / {ols_treatment_test:.4f}")

    print("  Boosting")
    boost_q = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_q.fit(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_train = boost_q.score(train_dict[feature_specification], train_dict["y_diff"])
    r2_boost_q_test  = boost_q.score(test_dict[feature_specification],  test_dict["y_diff"])
    print(f"  R2 Outcome diff train/test: {r2_boost_q_train:.4f} / {r2_boost_q_test:.4f}")

    boost_d = LGBMRegressor(n_estimators=500, learning_rate=0.02, random_state=42, verbose=-1)
    boost_d.fit(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_train = boost_d.score(train_dict[feature_specification], train_dict["d_diff"])
    r2_boost_d_test  = boost_d.score(test_dict[feature_specification],  test_dict["d_diff"])
    print(f"  R2 Treatment diff train/test: {r2_boost_d_train:.4f} / {r2_boost_d_test:.4f}")

    results_df_diff_tab = pd.concat([results_df_diff_tab, pd.DataFrame(data=[
        [ols_outcome_train, ols_outcome_test, ols_treatment_train, ols_treatment_test],
        [r2_boost_q_train,  r2_boost_q_test,  r2_boost_d_train,   r2_boost_d_test],
    ])], axis=0)

row_names = [
    "OLS (Tabular)", "Boosting (Tabular)",
    "OLS (Tabular + PCA)", "Boosting (Tabular + PCA)",
    "OLS (Tabular + Similarities)", "Boosting (Tabular + Similarities)",
    "OLS (Tabular + Embeddings)", "Boosting (Tabular + Embeddings)",
]
results_df_diff_tab.columns = column_names
results_df_diff_tab.index   = row_names
results_df_diff_tab


 Feature specification: x
  OLS
  R2 Outcome diff train/test: 0.3286 / 0.3550
  R2 Treatment diff train/test: 0.0152 / 0.0166
  Boosting


  R2 Outcome diff train/test: 0.5704 / 0.5224


  R2 Treatment diff train/test: 0.2077 / 0.0968

 Feature specification: x_pca
  OLS
  R2 Outcome diff train/test: 0.3302 / 0.3555
  R2 Treatment diff train/test: 0.0192 / 0.0178
  Boosting


  R2 Outcome diff train/test: 0.6480 / 0.5202


  R2 Treatment diff train/test: 0.3466 / 0.0513

 Feature specification: x_sim
  OLS
  R2 Outcome diff train/test: 0.3312 / 0.3549
  R2 Treatment diff train/test: 0.0195 / 0.0184
  Boosting


  R2 Outcome diff train/test: 0.6374 / 0.5227


  R2 Treatment diff train/test: 0.3365 / 0.0484

 Feature specification: x_emb
  OLS
  R2 Outcome diff train/test: 0.3492 / 0.3110
  R2 Treatment diff train/test: 0.0560 / -0.0573
  Boosting


  R2 Outcome diff train/test: 0.7400 / 0.5179


  R2 Treatment diff train/test: 0.5057 / -0.0100


,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.328582,0.355020,0.015228,0.016596
Boosting (Tabular),0.570397,0.522406,0.207716,0.096754
OLS (Tabular + PCA),0.330210,0.355492,0.019221,0.017814
Boosting (Tabular + PCA),0.647992,0.520214,0.346625,0.051349
OLS (Tabular + Similarities),0.331152,0.354852,0.019539,0.018356
Boosting (Tabular + Similarities),0.637399,0.522726,0.336455,0.048389
OLS (Tabular + Embeddings),0.349221,0.310958,0.055953,-0.057325
Boosting (Tabular + Embeddings),0.740020,0.517910,0.505697,-0.010040


## ⑮ Summary — Diff Models

In [15]:
results_df_diff = pd.concat([results_df_diff_tab, results_df_diff_deep], axis=0)
results_df_diff

,R2 Q Train,R2 Q Test,R2 P Train,R2 P Test
OLS (Tabular),0.328582,0.355020,0.015228,0.016596
Boosting (Tabular),0.570397,0.522406,0.207716,0.096754
OLS (Tabular + PCA),0.330210,0.355492,0.019221,0.017814
Boosting (Tabular + PCA),0.647992,0.520214,0.346625,0.051349
OLS (Tabular + Similarities),0.331152,0.354852,0.019539,0.018356
Boosting (Tabular + Similarities),0.637399,0.522726,0.336455,0.048389
OLS (Tabular + Embeddings),0.349221,0.310958,0.055953,-0.057325
Boosting (Tabular + Embeddings),0.740020,0.517910,0.505697,-0.010040
Deep Time Independent,0.004726,0.001411,-0.019750,-0.017220
Deep Time Dependent,0.012393,0.010064,-0.015636,-0.013574


## ⑯ Final Summary (% format)

R² multiplied by 100 — matches paper Table 2 format.
Compare with 03_2 (txt only) to see the gain from adding image embeddings.

In [16]:
print("=== Level Models — Test R² (%) ===")
print(results_df[["R2 Q Test", "R2 P Test"]].round(4) * 100)
print()
print("=== Diff Models — Test R² (%) ===")
print(results_df_diff[["R2 Q Test", "R2 P Test"]].round(4) * 100)

=== Level Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          28.88       2.51
Boosting (Tabular)                     63.08       7.66
OLS (Tabular + PCA)                    58.80       3.78
Boosting (Tabular + PCA)               74.01       4.14
OLS (Tabular + Similarities)           59.18       3.91
Boosting (Tabular + Similarities)      73.93      14.72
OLS (Tabular + Embeddings)             52.69     -39.97
Boosting (Tabular + Embeddings)        75.19       6.68
Deep Time Independent                  66.95      -6.74
Deep Time Dependent                    80.85      -8.46

=== Diff Models — Test R² (%) ===
                                   R2 Q Test  R2 P Test
OLS (Tabular)                          35.50       1.66
Boosting (Tabular)                     52.24       9.68
OLS (Tabular + PCA)                    35.55       1.78
Boosting (Tabular + PCA)               52.02       5.13
OLS (Tabular + Similarities)      